In [1]:
%reset -f

In [2]:
import openpyxl
from openpyxl.utils import column_index_from_string
import pandas as pd
import importlib
import os
from datetime import datetime
import json
import simulation_class  
import numpy as np
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.problems.functional import FunctionalProblem


importlib.reload(simulation_class)
HeatPumpSimulator = simulation_class.HeatPumpSimulator


## Extraction of the Output Data

Extraction of output data from the `All data` sheet of the Excel file and storing in a DataFrame `df_output`.

- Column K : Water out -> water temperature leaving the heat pump; FMU output
- Column O : Power out -> thermal power of the heat pump (hot side); FMU output
- Column P : Power in -> electrical power of the heat pump; FMU output
- Column Q : COP -> coefficient of performance of the heat pump; FMU output

In [3]:
excel_file_path = './../data/Databrut.xlsx'

workbook = openpyxl.load_workbook(excel_file_path)
sheet = workbook.active

# Définir les colonnes et les noms correspondants
columns_info = {
    "K": "Water_out",
    "O": "Power_out",
    "P": "Power_in",
    "Q": "COP",
    "M": "Defrosts"
}

stored_data = {name: [] for name in columns_info.values()}

# Itérer sur les colonnes et stocker les données dans le dictionnaire
for col_letter, col_name in columns_info.items():
    col_index = column_index_from_string(col_letter)
    if col_index <= sheet.max_column:
        for cell in sheet[col_letter]:
            stored_data[col_name].append(cell.value)
    else:
        print(f"Column {col_letter} not found in the sheet.")

workbook.close()

df_output = pd.DataFrame(stored_data)
df_output = df_output.drop([0, 1, 2, 3])  

for column in ["Water_out", "Power_out", "Power_in", "COP"]:
    df_output[column] = pd.to_numeric(df_output[column], errors='coerce')

df_output.dropna(subset=["Water_out", "Power_out", "Power_in", "COP"], inplace=True)
df_output['Water_out'] = df_output['Water_out'] + 273.15
df_output = df_output.rename(columns={'Water_out': 'y_T_out', 'Power_out': 'y_heatPower', 'Power_in': 'y_elecPower', 'COP': 'y_COP'})

df_output.head()


,y_T_out,y_heatPower,y_elecPower,y_COP,Defrosts
4,308.248729,8628.775284,2022.298343,4.279641,no
5,308.320939,8589.444308,2044.149171,4.214619,no
6,308.179724,7641.797307,1748.723757,4.383122,no
7,308.268453,6873.783742,1477.259669,4.667061,no
8,308.200442,6035.440249,1248.497238,4.848746,no


In [4]:
#Export l'output en csv
Outputpath = './../data/df_output.csv'
df_output.to_csv(Outputpath, index=False)

## Extraction of the Input Data

Extraction of the input data from the `All data` sheet of the Excel file and storing in a DataFrame `df_input`.

- Column I : Frequency -> compressor frequency; FMU input
- Column J : Water in -> water temperature at the heat pump inlet; FMU input
- Column L : Flow -> water flow; FMU input
- Column M : Defrosts -> indicates if defrosting took place during the test

In [5]:
excel_file_path = './../data/Databrut.xlsx'

workbook = openpyxl.load_workbook(excel_file_path)
sheet = workbook.active

columns_info = {
    "I": "Frequency",
    "J": "Water_in",
    "L": "Flow",
    "M": "Defrosts"  
}

stored_data = {name: [] for name in columns_info.values()}

for col_letter, col_name in columns_info.items():
    col_index = column_index_from_string(col_letter)
    if col_index <= sheet.max_column:
        for row in sheet.iter_rows(min_row=2, max_col=col_index, min_col=col_index, values_only=True):  
            stored_data[col_name].extend(row)
    else:
        print(f"Column {col_letter} not found in the sheet.")

workbook.close()

df_input = pd.DataFrame(stored_data)

for column in ["Frequency", "Water_in", "Flow"]:
    df_input[column] = pd.to_numeric(df_input[column], errors='coerce')

df_input.dropna(subset=["Frequency", "Water_in", "Flow"], inplace=True)


# Création de la colonne 'T_air' avec les valeurs correctes
NbA12W35,NbA12W45 = 9,9
NbA7W35,NbA7W45,NbA7W55 = 13,12,4
NbA2W35,NbA2W45,NbA2W55  = 5,5,5
NbAmoins7W35,NbAmoins7W45,NbAmoins7W55 = 5,5,5

NbA12 = NbA12W35 + NbA12W45
NbA7 = NbA7W35 + NbA7W45 + NbA7W55
NbA2 = NbA2W35 + NbA2W45 + NbA2W55
NbAmoins7 = NbAmoins7W35 + NbAmoins7W45 + NbAmoins7W55

values = [12]*NbA12 + [7]*NbA7 + [2]*NbA2 + [-7]*NbAmoins7

if len(values) == len(df_input):
    df_input['T_air'] = values
else:
    print("Erreur : La longueur des valeurs de 'T_air' ne correspond pas au DataFrame.")

columns_order = ["Frequency", "Water_in", "Flow", "T_air", "Defrosts"] 
df_input = df_input[columns_order]  # Réorganisation des colonnes

# df_input.head()


In [6]:
# Conversion de la température de l'eau de °C à K
df_input['Water_in'] = df_input['Water_in'] + 273.15

# Conversion du débit d'eau de m³/h à m³/s
df_input['Flow'] = df_input['Flow'] / 3600

# Conversion de la température de l'air de °C à K
df_input['T_air'] = df_input['T_air'] + 273.15

df_input.head()

,Frequency,Water_in,Flow,T_air,Defrosts
3,105.6,303.262818,0.000415,285.15,no
4,98.4,303.289006,0.000409,285.15,no
5,88.2,303.231215,0.000370,285.15,no
6,78.0,303.359724,0.000336,285.15,no
7,68.4,303.246077,0.000292,285.15,no


In [7]:
Inputpath = './../data/df_input.csv'
df_input.to_csv(Inputpath, index=False)


## Fisrt Simulation

Firstly, we ignore the tests where "Defrosts" = yes (column M)

In [8]:
df_input_no_defrost = df_input[df_input['Defrosts'] == 'no']
df_output_no_defrost = df_output[df_output['Defrosts'] == 'no']


In this notebook, we are importing the Python class `HeatPumpSimulator` that utilizes FMpy to simulate FMU. This class is responsible for performing the simulation of the heat pump system. Below are the defined inputs and outputs for the simulation.


#### FMU Inputs

- **u_compressorFrequency:** Compressor Frequency *Unit:* Hz

- **u_VFlow:** Water Flow *Unit:* m³/s

- **u_T_water:** Water Temperature Entering the Heat Pump *Unit:* K

- **u_T_air:** Air Temperature at the Heat Pump Inlet *Unit:* K

#### FMU Outputs

- **y_T_out:** Water Temperature Leaving the Heat Pump *Unit:* K

- **y_heatPower:** Thermal Power of the Heat Pump (Hot Side) *Unit:* W

- **y_elecPower:** Electrical Power of the Heat Pump *Unit:* W

- **y_COP:** Coefficient of Performance of the Heat Pump *Unit:* []


In [9]:
simulator = HeatPumpSimulator('../FMU/HPFMU_20_Linux.fmu')

last_rows = []

for index, row in df_input_no_defrost.head(10).iterrows():
    start_values = {
        'u_compressorFrequency': row['Frequency'],
        'u_VFlow': row['Flow'],
        'u_T_watrer_in': row['Water_in'],
        'u_T_air': row['T_air'],
        # Valeurs fixes des paramètres pour l'instant
        'x_areaLeakage': 5e-7,
        'x_areaSuctionValve': 5e-4,
        'x_areaDischargeValve': 5e-5,
        'x_relativeDeadSpace': 0.05,
        'x_driveEfficiency': 0.9
    }

    simulation_result = simulator.simulate(start_values)
    df_simulation_result = pd.DataFrame(simulation_result)
    last_row = df_simulation_result.iloc[-1]
    last_rows.append(last_row)


In [10]:
df_final_results = pd.DataFrame(last_rows)
df_final_results = df_final_results.drop(columns=['time'])
df_final_results = df_final_results.reset_index(drop=True)

df_final_results.head()


,y_T_out,y_heatPower,y_elecPower,y_COP
0,308.585764,9181.373552,2427.614040,3.782057
1,308.320366,8559.788081,2223.600652,3.849517
2,308.191957,7635.408713,1945.937162,3.923769
3,308.135888,6669.292055,1680.123506,3.969525
4,307.972277,5738.200241,1435.660306,3.996907


## Line by line calibration (defrost = no) 

We separate the data into two groups: a first group on which we will carry out the calibration and a second group on which we will carry out the validation.

We take a size of 90% for the first group and 10% for the second group.

In [11]:
size_train = 0.9

In [12]:
df_input_no_defrost_cal = df_input_no_defrost.sample(frac=size_train, random_state=0)
df_input_no_defrost_test = df_input_no_defrost.drop(df_input_no_defrost_cal.index)

df_output_no_defrost_cal = df_output_no_defrost.sample(frac=size_train, random_state=0)
df_output_no_defrost_test = df_output_no_defrost.drop(df_output_no_defrost_cal.index)

We create a fonction that allows us to save all the results of the calibration.

In [13]:
def save_results(results, filename_prefix="pso_results", size_train=0.9):
    try:
        current_script_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        current_script_dir = os.getcwd()

    results_dir = os.path.join(current_script_dir, "results_" + str(int(size_train * 100)))

    if not os.path.exists(results_dir):
        os.makedirs(results_dir)

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")  
    filename = f"{filename_prefix}_{timestamp}.json"
    filepath = os.path.join(results_dir, filename)

    
    for key, res in results.items():
        for subkey, value in res.items():
            if isinstance(value, np.ndarray):
                res[subkey] = value.tolist()

    
    with open(filepath, "w") as f:
        json.dump(results, f, indent=4)

    print(f"Results saved in {filepath}")

Now, for calibration, we use the NSGA2 algorithm from the `pymoo` library.

The objective function is defined as follows:
$$
 f(x) =
  \begin{cases}
    \infty      & \quad \text{if the simulation failts} \\
    \text{error}   & \quad \text{else } 
  \end{cases}
$$

With 
$$
f(x) = [f_1(x), f_2(x), f_3(x), f_4(x)] \quad \text{ and } \quad \text{error} = [\text{error}_{1}, \text{error}_{2}, \text{error}_{3}, \text{error}_{4}]
$$
And

$$
\text{error}_{1} = |\text{True}_{Tout}-\text{Fmu}_{Tout}|  \quad \text{ and } \quad
\text{error}_{2} = |\text{True}_{heatPower}-\text{Fmu}_{heatPower}|
$$

$$\text{error}_{3} =|\text{True}_{elecPower}-\text{Fmu}_{elecPower}|  \quad \text{ and } \quad \text{error}_{4} = |\text{True}_{COP}-\text{Fmu}_{COP}| $$



Simulation failed means that the execution time lasted more than 10 seconds or that the fmu stopped with an error.


In [14]:
infini = 10000000
def objective_T_out(params, line_index=0):
    print(f"Évaluation des paramètres pour la ligne {line_index + 1}")
    input_row = df_input_no_defrost_cal.iloc[line_index]
    output_row = df_output_no_defrost_cal.iloc[line_index]

    start_values = {
        'u_compressorFrequency': input_row['Frequency'],
        'u_VFlow': input_row['Flow'],
        'u_T_watrer_in': input_row['Water_in'],
        'u_T_air': input_row['T_air'],
        'x_areaLeakage': params[0],
        'x_areaSuctionValve': params[1],
        'x_areaDischargeValve': params[2],
        'x_relativeDeadSpace': params[3],
        'x_driveEfficiency': params[4]
    }

    simulation_result = simulator.simulate_with_timeout(
        start_values, timeout=10)
    if not simulation_result:
        print("La simulation a échoué ou a été interrompue.")
        return infini

    last_row = pd.DataFrame(simulation_result).iloc[-1]
    error = np.sum(abs(last_row[['y_T_out']] - [output_row['y_T_out']]))
    print(f"Erreur : {error}")

    return error

def objective_heatPower(params, line_index=0):
    print(f"Évaluation des paramètres pour la ligne {line_index + 1}")
    input_row = df_input_no_defrost_cal.iloc[line_index]
    output_row = df_output_no_defrost_cal.iloc[line_index]

    start_values = {
        'u_compressorFrequency': input_row['Frequency'],
        'u_VFlow': input_row['Flow'],
        'u_T_watrer_in': input_row['Water_in'],
        'u_T_air': input_row['T_air'],
        'x_areaLeakage': params[0],
        'x_areaSuctionValve': params[1],
        'x_areaDischargeValve': params[2],
        'x_relativeDeadSpace': params[3],
        'x_driveEfficiency': params[4]
    }

    simulation_result = simulator.simulate_with_timeout(
        start_values, timeout=10)
    if not simulation_result:
        print("La simulation a échoué ou a été interrompue.")
        return infini

    last_row = pd.DataFrame(simulation_result).iloc[-1]
    error = np.sum(abs(last_row[['y_heatPower']] - [output_row['y_heatPower']]))
    print(f"Erreur : {error}")

    return error

def objective_elecPower(params, line_index=0):
    print(f"Évaluation des paramètres pour la ligne {line_index + 1}")
    input_row = df_input_no_defrost_cal.iloc[line_index]
    output_row = df_output_no_defrost_cal.iloc[line_index]

    start_values = {
        'u_compressorFrequency': input_row['Frequency'],
        'u_VFlow': input_row['Flow'],
        'u_T_watrer_in': input_row['Water_in'],
        'u_T_air': input_row['T_air'],
        'x_areaLeakage': params[0],
        'x_areaSuctionValve': params[1],
        'x_areaDischargeValve': params[2],
        'x_relativeDeadSpace': params[3],
        'x_driveEfficiency': params[4]
    }

    simulation_result = simulator.simulate_with_timeout(
        start_values, timeout=10)
    if not simulation_result:
        print("La simulation a échoué ou a été interrompue.")
        return infini

    last_row = pd.DataFrame(simulation_result).iloc[-1]
    error = np.sum(abs(last_row[['y_elecPower']] - [output_row['y_elecPower']]))
    print(f"Erreur : {error}")

    return error

def objective_COP(params, line_index=0):
    print(f"Évaluation des paramètres pour la ligne {line_index + 1}")
    input_row = df_input_no_defrost_cal.iloc[line_index]
    output_row = df_output_no_defrost_cal.iloc[line_index]

    start_values = {
        'u_compressorFrequency': input_row['Frequency'],
        'u_VFlow': input_row['Flow'],
        'u_T_watrer_in': input_row['Water_in'],
        'u_T_air': input_row['T_air'],
        'x_areaLeakage': params[0],
        'x_areaSuctionValve': params[1],
        'x_areaDischargeValve': params[2],
        'x_relativeDeadSpace': params[3],
        'x_driveEfficiency': params[4]
    }

    simulation_result = simulator.simulate_with_timeout(
        start_values, timeout=10)
    if not simulation_result:
        print("La simulation a échoué ou a été interrompue.")
        return infini

    last_row = pd.DataFrame(simulation_result).iloc[-1]
    error = np.sum(abs(last_row[['y_COP']] - [output_row['y_COP']]))
    print(f"Erreur : {error}")

    return error

In [15]:
def apply_NSGA2(df_input_no_defrost_cal, pop_size=20, n_gen = 15):
    # For each row of the training data, we apply NSGA-II
    nga2_results = {}
    n_train = len(df_input_no_defrost_cal)
    for line_index in range(n_train):
        print(f"Calibration pour la ligne {line_index + 1}")
        # Define the objective function 
        objs = [lambda x: objective_T_out(x, line_index=line_index),
                lambda x: objective_heatPower(x, line_index=line_index),
                lambda x: objective_elecPower(x, line_index=line_index),
                lambda x: objective_COP(x, line_index=line_index)
                ]

        # Define the problem
        my_problem= FunctionalProblem(n_var=5,objs = objs,
                                xl=np.array([1e-8, 2e-6,1e-7 , 0.001, 0.85]),
                                xu=np.array([1e-6,1e-3,1e-4 ,0.1,0.98]))
        
        # Optimize the problem using NSGA-II
        algorithm = NSGA2(pop_size=pop_size)
        result = minimize(my_problem,
                        algorithm,
                        ('n_gen', n_gen),
                        seed=1,
                        verbose=True)
        
        nga2_results[line_index] = {
            "xopt": result.X,
            "fopt": result.F
        }
        
        # Display the results
        print("Optimal solutions:")
        print(result.X)
        print("Optimal objectives:")
        print(result.F)
        save_results(nga2_results, filename_prefix="nga2_results_defrost_no")
    
    return 0

We export the `json` results file we want to use for the validation.

In [14]:
with open('results_90/nga2_results_defrost_no_2024-01-29_20-32-22.json', 'r') as f:
    optimized_params = json.load(f)

In [23]:
for k, v in optimized_params.items():
    print(v['fopt'])
    break;

[[0.3709826091447894, 501.2022132543825, 2.7026588406756673, 0.3339465113935449], [0.21635225007491954, 239.54404391698426, 80.37496543301495, 0.012884106061858525], [0.1908844105890921, 207.42176514709263, 89.90155811628301, 0.05181405002465134], [0.014573928623804022, 14.941110772899265, 25.412265503973686, 0.0627292097528116], [0.2090040966355673, 230.2962557633482, 78.49433151393714, 0.014870167814856838], [0.0018033430037576181, 31.04647438532993, 61.38668083719608, 0.14491104105521657], [0.014032814374104419, 15.623476571937317, 37.41652519242098, 10000000.0], [0.1433898694623963, 214.16572290477416, 3.9710124047312547, 0.14854397698065291], [0.19141225045535748, 208.088739607143, 77.49242439941145, 0.026718231322829578], [10000000.0, 19.303157759218266, 24.986976976797905, 0.06467316002449186]]
